In [2]:
import pandas as pd
import numpy as np
import random

In [108]:
def get_entropy(y: pd.Series):
    unique_labels = y.unique()
    
    if len(unique_labels) == 1:
        return 0
    
    n = len(y)
    count_1 = y.sum()
    count_0 = n - count_1

    proba_0 = count_0 / n
    proba_1 = count_1 / n

    return - (np.log2(proba_0) * proba_0 + np.log2(proba_1) * proba_1)
       
def get_best_split(X: pd. DataFrame, y : pd.Series):

    igs = []
    
    for column in X.columns:
        sorted_unique_values = sorted(X[column].unique())
        entropy = get_entropy(y)
        m = len(sorted_unique_values)
        
        for i in range(m - 1):
            split_value = (sorted_unique_values[i] + sorted_unique_values[i + 1]) / 2 
            
            left_indexes = X.index[X[column] <= split_value]
            left_y = y.loc[left_indexes]
            left_entropy = get_entropy(left_y)
            
            right_indexes = X.index[X[column] > split_value]
            right_y = y.loc[right_indexes]
            right_entropy = get_entropy(right_y)

            ig = entropy - left_entropy * len(left_indexes) / len(y) - right_entropy * len(right_indexes) / len(y) 

            igs.append((column, split_value, ig))

    return max(igs, key= lambda x: x[2])



class Leaf:
    def __init__(self, parent, samples, proba, depth):
        self.parent = parent
        self.samples = samples
        self.proba = proba
        self.depth = depth

    def __str__(self):
        return f'{"  " * self.depth} {self.proba}'

class Node:
    def __init__(self, parent=None):
        self.parent = parent
        self.left = None
        self.right = None
        self.samples = None
        self.depth = 1
        self.split_value = None
        self.column = None
        self.ig = None
        

    def create_leafs(self, X, y):
        self.column, self.split_value, self.ig = get_best_split(X, y)
        
        left_indexes = X.index[X[self.column] <= self.split_value]
        left_y = y.loc[left_indexes]
        self.left = Leaf(self, len(left_y), left_y.mean(), self.depth + 1)
        
        right_indexes = X.index[X[self.column] > self.split_value]
        right_y = y.loc[right_indexes]
        self.right = Leaf(self, len(right_y), right_y.mean(), self.depth + 1)

    def __str__(self):
        return  '\n'.join([
        f'{"  " * self.depth} {self.column} {self.split_value}',
        f'{self.left}',
        f'{self.right}'])

    def predict_proba(self, row: pd.Series):
        if row.loc[self.column] <= self.split_value:
            if isinstance(self.left, Leaf):
                return self.left.proba
            else: 
                return self.left.predict_proba(row)
        else:
            if isinstance(self.right, Leaf):
                return self.right.proba
            else: 
                return self.right.predict_proba(row)

class MyTreeClf:
    def __init__(self, max_depth: int = 5, min_samples_split: int = 2,
                 max_leafs: int = 20, bins=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_leafs = max_leafs
        self.nodes_cnt = 1
        self.leafs_cnt = None
        self.bins = bins
        

    def __str__(self):
        return f"MyTreeClf class: max_depth={self.max_depth}, min_samples_split={self.min_samples_split}, max_leafs={self.max_leafs}"

    def fit(self, X: pd.DataFrame, y: pd.Series):
        self.root = Node()
        self.build_tree_recursive(self.root, X, y)
        self.leafs_cnt = self.nodes_cnt + 1

    def predict_proba(self, X: pd.DataFrame) -> pd.Series:
        probas = []
        for index, row in X.iterrows():
            proba = self.root.predict_proba(row)
            probas.append(proba)
        return pd.Series(probas, X.index)

    def predict(self, X: pd.DataFrame) -> pd.Series:
        probas = self.predict_proba(X)
        return probas > 0.5
        
    def build_tree_recursive(self, node: Node, X, y):
        if node.depth == self.max_depth:
            node.create_leafs(X, y)
            return     
        if self.nodes_cnt + 1 >= self.max_leafs:
            node.create_leafs(X, y)
            return 

        node.column, node.split_value, node.ig = get_best_split(X, y)
        
        left_indexes = X.index[X[node.column] <= node.split_value]
        left_cnt = len(left_indexes)
        left_y = y.loc[left_indexes]
        
        if len(left_y.unique()) == 1 or left_cnt < self.min_samples_split:
            node.left = Leaf(node, len(left_y), left_y.mean(), node.depth + 1)
        else:
            node.left = Node(node)
            node.left.depth = node.depth + 1
            self.nodes_cnt += 1
            self.build_tree_recursive(node.left, X.loc[left_indexes], left_y)

        right_indexes = X.index[X[node.column] > node.split_value]
        right_cnt = len(right_indexes)
        right_y = y.loc[right_indexes]
        
        if len(right_y.unique()) == 1  or right_cnt < self.min_samples_split or self.nodes_cnt + 1 >= self.max_leafs:
            node.right = Leaf(node, len(right_y), right_y.mean(), node.depth + 1)
        else:
            node.right = Node(node)
            node.right.depth = node.depth + 1
            self.nodes_cnt += 1
            self.build_tree_recursive(node.right, X.loc[right_indexes], right_y)
        
                
        

In [109]:

df = pd.read_csv('./data_banknote_authentication.txt', header=None)
df.columns = ['variance', 'skewness', 'curtosis', 'entropy', 'target']
X, y = df.iloc[:,:4], df['target']

In [110]:
my_tree_clf = MyTreeClf(5,2,1)
my_tree_clf.fit(X, y)

In [111]:
my_tree_clf.roo(X)

0       False
1       False
2       False
3       False
4       False
        ...  
1367    False
1368     True
1369     True
1370     True
1371     True
Length: 1372, dtype: bool